# Lightweight MI Models
**STOPPED FIXED SETTINGS.** FBLightConvNet and EEGITNet are complete negative results; SincShallowNet is implementation-only. Do not repeat this sweep as an accuracy follow-up; see `AGENTS.md` section 2e. Execution fails closed.

# 1. Setup

In [ ]:
from __future__ import annotations
raise RuntimeError('CLOSED lightweight-model sweep: see AGENTS.md section 2e.')
import builtins, hashlib, json, os, platform, random, sys, time
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from modern_mi_common import *
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | CWD: {Path.cwd()}")

# 2. Configuration
## 2.1 Liu2024 Defaults
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # Paths / run identity
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-lightweight-mi-models"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "lightweight_mi_locked_fblightconvnet",
    "config_note": "Locked exactly-once lightweight target-only benchmark.",
    # Dataset and independent trial preprocessing
    "subjects_to_use": None, "channel_set": "liu29", "native_sfreq": 500, "target_sfreq": 128,
    "marker_channel_index": 32, "onset_marker_value": 2, "onset_plausible_range": [800, 1300],
    "window_seconds": 4.0, "bandpass_hz": [4.0, 40.0], "filter_order": 4,
    "normalization_mode": "channel_standardize", "normalization_eps": 1e-6,
    # Model / evaluation
    "model_name": "FBLightConvNet", "allow_optional_models": False, "model_kwargs": {"win_len": 256},
    "optional_model_kwargs": {"EEGInceptionMI": {"n_convs": 3, "n_filters": 16}, "MSVTNet": {"n_filters_list": [4,4,4,4], "num_layers": 1}, "CTNet": {"embed_dim": 20, "num_layers": 2}},
    "evaluation_mode": "stratified_5fold", "cv_folds": 5, "cv_random_state": 2026,
    "expected_global_split_hash": "801ec1d2c981335f",
    # Fixed direct-comparison optimization
    "batch_size": 8, "n_epochs": 20, "learning_rate": 0.0003, "weight_decay": 0.01, "gradient_clip_norm": 1.0,
    "seeds": [2026], "seed": 2026, "set_seed": True, "bootstrap_iterations": 10000, "collapse_threshold": 0.9,
    # Completed exactly-once ShallowFBCSP reference
    "shallow_reference_artifact": str(WORKING_DIR / "artifacts" / "liu2024-compact-mi-models" / "20260712_165746_790145_fd8ab986" / "subject_metrics.json")
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp=datetime.now().strftime("%Y%m%d_%H%M%S_%f"); config_hash=hashlib.md5(json.dumps(CONFIG,sort_keys=True,default=str).encode()).hexdigest()[:8]; return f"{timestamp}_{config_hash}"
RUN_ID=create_run_id(); ARTIFACT_DIR=Path(CONFIG["artifact_dir"])/RUN_ID; ARTIFACT_DIR.mkdir(parents=True,exist_ok=False)
LOG_PATH=ARTIFACT_DIR/"run.log"; _LOG_FILE_HANDLE=open(LOG_PATH,"a",buffering=1,encoding="utf-8",errors="replace")
def _safe_write_text(stream,text):
    try: stream.write(text)
    except UnicodeEncodeError:
        enc=getattr(stream,"encoding",None) or "utf-8"; stream.write(text.encode(enc,errors="replace").decode(enc,errors="replace"))
def _timestamped_print(*args,**kwargs):
    sep=kwargs.pop("sep"," " ); end=kwargs.pop("end","\n"); flush=kwargs.pop("flush",False); file=kwargs.pop("file",None); target=sys.stdout if file is None else file; message=sep.join(str(a) for a in args); stamped=f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}"; _safe_write_text(target,stamped+end); _safe_write_text(_LOG_FILE_HANDLE,stamped+end)
    if flush: target.flush(); _LOG_FILE_HANDLE.flush()
builtins.print=_timestamped_print
config_path=ARTIFACT_DIR/"config.json"; config_path.write_text(json.dumps(CONFIG,indent=2),encoding="utf-8")
print(f"Run ID:     {RUN_ID}"); print(f"Artifacts:  {ARTIFACT_DIR}"); print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")
DEVICE=resolve_device(); print(f"Using device: {DEVICE}")
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"]=str(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True,warn_only=True)
BASE_SEED=int(CONFIG["seed"]); seed_everything(BASE_SEED)

# 3. Load and Prepare Data
## 3.1 Data Loading Helpers
Uses `modern_mi_common.py`: Liu29 ordering, independent full-trial filtering, marker crop, and 128 Hz resampling.
## 3.2 Fold-Local Normalization
## 3.3 Dataset Classes
## 3.4 Locate and Load Data

# 4. Model
## 4.1 Build Model and Trainable Parameters
Primary models are FBLightConvNet, SincShallowNet, and EEGITNet. Reduced optional models require both installation and `allow_optional_models=true`.
## 4.2 Architecture Guards
FBLightConvNet `win_len` must divide 512 exactly; 256 is canonical and 128 is supported.

In [ ]:
def resolved_model_kwargs():
    name=CONFIG["model_name"]; primary=set(LIGHTWEIGHT_MODELS); optional=set(LIGHTWEIGHT_OPTIONAL_MODELS)
    if name not in primary|optional: raise ValueError(f"Unsupported lightweight model: {name}")
    if name in optional and not CONFIG["allow_optional_models"]: raise ValueError(f"{name} is optional; set allow_optional_models=true explicitly")
    from braindecode import models
    if not hasattr(models,name): raise ImportError(f"Braindecode model {name} is not installed")
    kwargs=dict(CONFIG.get("model_kwargs",{})); kwargs.update(CONFIG.get("optional_model_kwargs",{}).get(name,{}) if name in optional else {})
    if name=="FBLightConvNet":
        win=int(kwargs.get("win_len",256))
        if win not in {128,256} or 512%win: raise ValueError("FBLightConvNet win_len must be 128 or 256 and divide 512")
        kwargs["win_len"]=win
    return kwargs
def train_with_curves(model,x_train,y_train,x_test,cfg,seed):
    seed_everything(seed); device=next(model.parameters()).device; loader=DataLoader(TensorDataset(torch.from_numpy(x_train),torch.from_numpy(y_train).long()),batch_size=cfg["batch_size"],shuffle=True,generator=torch.Generator().manual_seed(seed)); opt=torch.optim.AdamW(model.parameters(),lr=cfg["learning_rate"],weight_decay=cfg["weight_decay"]); curve=[]; start=time.perf_counter()
    for epoch in range(int(cfg["n_epochs"])):
        model.train(); total=0.0; count=0
        for xb,yb in loader:
            opt.zero_grad(set_to_none=True); loss=nn.functional.cross_entropy(model(xb.to(device)),yb.to(device)); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),cfg["gradient_clip_norm"]); opt.step(); total+=float(loss.detach())*len(xb); count+=len(xb)
        curve.append(total/count)
    model.eval()
    with torch.no_grad(): prob=model(torch.from_numpy(x_test).to(device)).softmax(1).cpu().numpy()
    return prob,curve,time.perf_counter()-start
def canonical_split_hash(all_splits): return stable_hash(all_splits)

# 5. Training
## 5.1 Fixed Last-Epoch Classifier
## 5.2 Within-Subject Exactly-Once Cross-Validation
## 5.3 Run All Subjects

In [ ]:
print("Full CONFIG banner:\n"+json.dumps(CONFIG,indent=2)); kwargs=resolved_model_kwargs()
paths=locate_subject_files(CONFIG["source_extract_dir"],CONFIG["subjects_to_use"]); SUBJECTS=[subject_id(p) for p in paths]; loaded={}; all_splits={}; inventory=[]
for p in paths:
    sid=subject_id(p); x,y,CH_NAMES,onsets=load_subject(p,CONFIG); splits=make_splits(y,CONFIG,sid); loaded[sid]=(x,y); all_splits[sid]=splits; inventory.append({"subject_id":sid,"path":str(p),"n_trials":len(y),"subject_split_hash":stable_hash(splits)})
split_hashes={stable_hash(v) for v in all_splits.values()}
if split_hashes!={CONFIG["expected_global_split_hash"]}: raise AssertionError(f"Subject split hashes {split_hashes} != locked {CONFIG['expected_global_split_hash']}")
GLOBAL_SPLIT_HASH=CONFIG["expected_global_split_hash"]
FOLD_RESULTS=[]
for sid in SUBJECTS:
    x,y=loaded[sid]
    for split in all_splits[sid]:
        tr=np.asarray(split["train_indices"]); te=np.asarray(split["test_indices"]); mean,scale=fit_normalizer(x[tr],CONFIG["normalization_mode"],CONFIG["normalization_eps"]); xtr=((x[tr]-mean)/scale).astype("float32"); xte=((x[te]-mean)/scale).astype("float32"); probs=[]; curves=[]; elapsed=0.0; model=None
        for seed in CONFIG["seeds"]:
            model=build_model(CONFIG["model_name"],x.shape[1],x.shape[2],CONFIG["target_sfreq"],DEVICE,kwargs); prob,curve,seconds=train_with_curves(model,xtr,y[tr],xte,CONFIG,seed); probs.append(prob); curves.append(curve); elapsed+=seconds
        prob=np.mean(probs,axis=0); pred=prob.argmax(1); FOLD_RESULTS.append(fold_result(sid,split["fold_id"],te,y[te],pred,prob,model,elapsed,BASE_SEED,{"seed_ensemble":CONFIG["seeds"],"training_loss_curves":curves,"split_hash":stable_hash(split),"global_split_hash":GLOBAL_SPLIT_HASH,"model_kwargs":kwargs}))
    observed=sorted(i for r in FOLD_RESULTS if r["subject_id"]==sid for i in r["test_indices"])
    if observed!=list(range(40)): raise AssertionError(f"{sid}: not exactly one OOF prediction per original trial")
subject_inventory_path=ARTIFACT_DIR/"subject_inventory.csv"; pd.DataFrame(inventory).to_csv(subject_inventory_path,index=False); (ARTIFACT_DIR/"splits.json").write_text(json.dumps(all_splits,indent=2),encoding="utf-8")

# 6. Results
## 6.1 Aggregate Trial-Level Metrics

In [ ]:
SUBJECT_ROWS=[]
for sid in SUBJECTS:
    rr=[r for r in FOLD_RESULTS if r["subject_id"]==sid]; order=np.argsort(np.concatenate([r["test_indices"] for r in rr])); yt=np.concatenate([r["true_labels"] for r in rr])[order]; yp=np.concatenate([r["predictions"] for r in rr])[order]; SUBJECT_ROWS.append({"subject_id":sid,"accuracy":float(accuracy_score(yt,yp)),"balanced_accuracy":float(balanced_accuracy_score(yt,yp)),"n_trials":40})
vals=[r["balanced_accuracy"] for r in SUBJECT_ROWS]; GLOBAL_METRICS={"mean_subject_balanced_accuracy":float(np.mean(vals)),"subject_bootstrap_95_ci":bootstrap_ci(vals,BASE_SEED,CONFIG["bootstrap_iterations"]),"n_subjects":len(vals),"n_folds_total":len(FOLD_RESULTS),"global_split_hash":GLOBAL_SPLIT_HASH,"collapse_rate":float(np.mean([r["collapse_diagnostics"]["collapsed"] for r in FOLD_RESULTS])),"parameter_count":FOLD_RESULTS[0]["parameter_count"]}
ref=Path(CONFIG["shallow_reference_artifact"])
if ref.exists():
    rdf=pd.DataFrame(json.loads(ref.read_text(encoding="utf-8"))); merged=pd.DataFrame(SUBJECT_ROWS).merge(rdf[["subject_id","balanced_accuracy"]],on="subject_id",suffixes=("","_shallow"),validate="one_to_one"); stat,p=wilcoxon(merged["balanced_accuracy"],merged["balanced_accuracy_shallow"]); delta=merged["balanced_accuracy"]-merged["balanced_accuracy_shallow"]; GLOBAL_METRICS["paired_vs_locked_shallowfbcsp"]={"mean_delta":float(delta.mean()),"wilcoxon_statistic":float(stat),"p_value_exploratory_uncorrected":float(p),"reference_path":str(ref)}

## 6.2 Performance Visualizations

In [ ]:
plot_path=ARTIFACT_DIR/"lightweight_mi_subject_performance.png"; plt.figure(figsize=(10,3)); plt.bar([r["subject_id"] for r in SUBJECT_ROWS],[100*r["balanced_accuracy"] for r in SUBJECT_ROWS]); plt.axhline(50,color="k",ls="--"); plt.xticks(rotation=90); plt.tight_layout(); plt.savefig(plot_path,dpi=150); plt.close()
loss_path=ARTIFACT_DIR/"training_loss_curves.png"; plt.figure(figsize=(6,3)); [plt.plot(c,alpha=.15,color="tab:blue") for r in FOLD_RESULTS for c in r["training_loss_curves"]]; plt.xlabel("Epoch"); plt.ylabel("Training loss"); plt.tight_layout(); plt.savefig(loss_path,dpi=150); plt.close()

## 6.3 Experiment Summary
## 6.4 Model Diagnostics

In [ ]:
print(json.dumps(GLOBAL_METRICS,indent=2))

## 6.5 Save Artifacts

In [ ]:
cv_results_path=ARTIFACT_DIR/"cv_results.json"; cv_results_path.write_text(json.dumps(FOLD_RESULTS,indent=2),encoding="utf-8")
subject_metrics_path=ARTIFACT_DIR/"subject_metrics.json"; subject_metrics_path.write_text(json.dumps(SUBJECT_ROWS,indent=2),encoding="utf-8")
global_metrics_path=ARTIFACT_DIR/"global_metrics.json"; global_metrics_path.write_text(json.dumps(GLOBAL_METRICS,indent=2),encoding="utf-8")
pd.DataFrame(SUBJECT_ROWS).to_csv(ARTIFACT_DIR/"subject_results.csv",index=False)
run_metadata={"run_id":RUN_ID,"artifact_dir":str(ARTIFACT_DIR),"experiment_name":CONFIG["experiment_name"],"config_note":CONFIG["config_note"],"subjects":SUBJECTS,"channel_names":CH_NAMES,"model_name":CONFIG["model_name"],"model_kwargs":kwargs,"seed":BASE_SEED,"global_split_hash":GLOBAL_SPLIT_HASH,"global_metrics":GLOBAL_METRICS,"artifacts":{p.name:str(p) for p in ARTIFACT_DIR.iterdir()}}
run_metadata_path=ARTIFACT_DIR/"run_metadata.json"; run_metadata_path.write_text(json.dumps(run_metadata,indent=2),encoding="utf-8")
print(f"CV results saved to:      {cv_results_path}"); print(f"Subject metrics saved to: {subject_metrics_path}"); print(f"Global metrics saved to:  {global_metrics_path}"); print(f"Run metadata saved to:    {run_metadata_path}"); print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass